In [1]:
!pip install "transformers<5" torch sentencepiece
! pip install youtube-transcript-api
! pip install torch
!pip install faiss-gpu

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 110.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 48.8 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.20.1
    Uninstalling huggingface_hub-1.20.1:
      Successfully uninstalled huggingface_hub-1.20.1
  Attempting uninstall: transformers
    Found existing installation: transformers 5.12.1
    Uninstalling transformers-5.12.1:
      Successfully uninstalled transformers-5.12.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.19.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.2/485.2 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━

In [20]:
from urllib.parse import urlparse, parse_qs
from youtube_transcript_api import YouTubeTranscriptApi
from transformers import pipeline
from sentence_transformers import SentenceTransformer
import re
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
import faiss
qwen_model_name  = "Qwen/Qwen2.5-3B-Instruct"
qwen_tokenizer  = AutoTokenizer.from_pretrained(qwen_model_name)
qwen_model  = AutoModelForCausalLM.from_pretrained(qwen_model_name, torch_dtype=torch.float16, device_map="auto")

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [3]:
def extract_video_id(url: str) -> str:
    parsed = urlparse(url)
    qs = parse_qs(parsed.query)
    video_ids = qs.get('v')

    if not video_ids:
        raise ValueError(f"No video id found in URL: {url}")
    return video_ids[0]

urllink = input("Enter Url of the Video :\n")
if __name__ == "__main__":
    url = urllink

    video_id = extract_video_id(url)

    api = YouTubeTranscriptApi()
    fetched = api.fetch(video_id, languages=['en'])

    yttext = "\n".join(snippet.text for snippet in fetched)

print(yttext)

[Music]
[Music]
Hey, hey, welcome to Honesty Wireless.
World's most honest salesman here. How
can I help you look familiar? You are
You're the guy who bought the iPhone 16,
aren't you? Yep, that was me. Told you
not to do it.
>> You did. You sure did. Yeah. So, what
brings you back? Need more perfectly
honest advice? Well, I actually have
good news and bad news. So, bad news is
uh I I broke the 16E. I sat on it. It's
a goner. Don't worry about it. Cuz the
good news is I won the lottery.
Oh.
Oh. Oh my god. Yeah. Pretty sick. Now,
keep it on the DL. You know, I'm not
telling anybody, but I'm telling you
because I can now buy whatever phone I
want. Money, no object. Oh, okay. Yeah,
great. Money, no object. Yeah, it's
happening. I'm the dream customer for
you right now. I will literally buy
whatever phone you tell me to. Just I
want one of the iPhones. So, what's it
going to be?
Okay. Wow. Yeah. So, you
you should order an iPhone
17.
Not the 17 Pro. Nope. Just 17.
So, so not the pro.
>> No

In [19]:
summarizer = pipeline("summarization", model="facebook/bart-large-cnn")
tokenizer = summarizer.tokenizer
model = summarizer.model 

Device set to use cuda:0


In [5]:
tokenizer = summarizer.tokenizer
model = summarizer.model

inputs = tokenizer(
    yttext,
    return_tensors="pt",
    truncation=True,
    max_length=1024
)
inputs = {k: v.to(model.device) for k, v in inputs.items()}
summary_ids = model.generate(
    inputs["input_ids"],
    attention_mask=inputs["attention_mask"],
    max_length=1000,
    min_length=900,
    do_sample=False
)

summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
print(summary)

Apple's new iPhone 17 Pro has a top tier display, variable refresh rate and PWM dimming. It also has a new 24 megapixel selfie camera. Apple's new point-and-shoot camera is "basically where everyone else should be calling it," says CNN's John Sutter. The new iPhone is on sale now for $1,000 more than last year's model, the 16E. It's available in black, white, red, blue, green, yellow and white. It goes on sale in the U.S. on Friday, September 14. For more information, go to Apple.com or watch CNN.com/soulmatestories. For confidential support on suicide matters call the Samaritans on 08457 90 90 90, visit a local Samaritans branch or see www.samaritans.org for details. In the United States, call the National Suicide Prevention Lifeline at 1-800-273-8255 or visit http://www.suicidepreventionlifeline.org/. For confidential. support in the UK, contact the National suicide Prevention Line on 1-856-788-7457. For information on the University of London's Samaritans, call 08457 909090 or visit

In [6]:
answer = summarizer(summary, max_length=130, min_length=30, do_sample=False)
answer

[{'summary_text': "Apple's new iPhone 17 Pro has a top tier display, variable refresh rate and PWM dimming. It also has a new 24 megapixel selfie camera. The new iPhone is on sale now for $1,000 more than last year's model, the 16E."}]

In [7]:
print(answer[0]['summary_text'])

Apple's new iPhone 17 Pro has a top tier display, variable refresh rate and PWM dimming. It also has a new 24 megapixel selfie camera. The new iPhone is on sale now for $1,000 more than last year's model, the 16E.


In [ ]:
"""
def extract_text(text):
    full_text  = ""
    for i in text:
        full_text += extract_text() + "\n"
    return full_text
    """

'\ndef extract_text(text):\n    full_text  = ""\n    for i in text:\n        full_text += extract_text() + "\n"\n    return full_text\n    '

In [8]:
def chunks_text(yttext , chunk_size=500 , overlap=50):
    words = yttext.split()
    chunks = []
    for i in range(0, len(words), chunk_size - overlap):
        chunk = " ".join(words[i:i + chunk_size])
        chunks.append(chunk)
    return chunks

In [9]:
def clean_text(yttext):
    text = yttext.replace("\n", " ")
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [10]:
def embed_chunks(chunks, model_name='sentence-transformers/all-MiniLM-L6-v2'):
    model = SentenceTransformer(model_name)
    embeddings = model.encode(chunks, convert_to_numpy=True)
    return model, embeddings

In [11]:
def create_faiss_index(embeddings):
    dim = embeddings.shape[1]
    index = faiss.IndexFlatL2(dim)
    index.add(embeddings)
    return index

In [12]:
def search_index(query, model, index, chunks, k=5):
    query_embedding = model.encode([query], convert_to_numpy=True)
    distances, indices = index.search(query_embedding, k)
    return [chunks[i] for i in indices[0]]

In [ ]:
def generate_text(prompt, max_length=100, num_return_sequences=1):
    inputs = tokenizer(prompt, return_tensors="pt")
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    print(inputs["input_ids"].device)
    print(model.device)
    outputs = model.generate(
        **inputs,
        max_length=max_length,
        num_return_sequences=num_return_sequences,
        do_sample=False,
        top_k=50,
        top_p=0.95,
    )
    return [tokenizer.decode(output, skip_special_tokens=True) for output in outputs]

In [14]:
text = clean_text(yttext)
chunks = chunks_text(yttext,chunk_size=120, overlap=30)
model_embeddings, embeddings = embed_chunks(chunks)
index = create_faiss_index(embeddings)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [15]:
Q = input("Enter your question :\n")
question = Q
top_chunks = search_index(question, model_embeddings, index, chunks, k=3)
context = "\n\n".join(top_chunks)
for i, chunk in enumerate(top_chunks, 1):
    print(f"\n Answer {i} \n{chunk}")


 Answer 1 
been more true about the iPhones. This 17 is really good. Why would you spend $300 extra dollars for a Pro? What are you getting out of that? You get the vapor chamber and the telephoto camera. And the more I thought about it, the more I think it actually means the Pro phones are actually appropriately named because the things you're getting out of those phones now for spending that extra money are a little smaller things that really only pros are going to notice or need. like the Pro chip with the better GPU and improved sustained thermal performance thanks to a vapor chamber. Most regular use of a smartphone is bursty and very on and off. But,

 Answer 2 
bracket where a lot of phones do actually still have telephoto cameras, which can be very useful. And maybe most glaringly obviously, no real improvements to Apple Intelligence still in the last couple months for any of the iPhones. But that's really about it. And a few of those things can be updated with software update

In [ ]:
https://www.youtube.com/watch?v=rng_yUSwrgU

In [ ]:
chunk = top_chunks[:3]
prompt = f"""
You are a question answering assistant.

Answer ONLY using the provided context.
If the answer is not in the context, reply exactly:
"I don't know."

Context:
{context}

Question:
{question}

Answer:
"""
answer = generate_text(prompt, max_length=700)
print(answer[0])

cuda:0
cuda:0
The new iPhone 17 is actually a good deal now, says John Sutter. Sutter: Most of my complaints complaints with the iPhone 17 are minor. The design obviously really didn't change at all since the 16, which means it's still matte all the way around the back and the sides, which I like.
